In [1]:
import time
import cv2
import numpy as np
import keyboard
from sdlarch_rl.utils.utils import FrameSkip
from stable_baselines3.common.atari_wrappers import WarpFrame
from stable_baselines3.common.env_util import make_vec_env
from notebooks.resident4 import RE4Env
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from pathlib import Path
import pygame
from inputs import get_gamepad
from inputs import devices
import threading
from imitation.data import serialize
from imitation.data.types import Trajectory
import torch as th
import os
import re
import random

from gymnasium.wrappers import FrameStack

MAX_TRAJ=10

demo_path = 'demos/'

os.makedirs(demo_path, exist_ok=True)

last_index = -1

for p in Path(demo_path).iterdir():
    m = re.search(r"demos(\d+)\.pt$", p.name)
    if m:
        num = int(m.group(1))
        last_index = max(last_index, num)

print("last_index: " + str(last_index))

myEnv = None

def make_env():
    def _init():
        global myEnv
        myEnv = RE4Env()

        env = WarpFrame(myEnv, width=128, height=128)
        # env = FrameSkip(env, skip=4)
        
        return env
    return _init

env = make_vec_env(make_env(), n_envs=1, vec_env_cls=DummyVecEnv)
env = VecFrameStack(env, 4)

SCALE = 3
SCREEN_WIDTH = 640*SCALE
SCREEN_HEIGHT = 480*SCALE

pygame.init()

# window = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))

window = pygame.display.set_mode(
    (SCREEN_WIDTH, SCREEN_HEIGHT),
    pygame.HWSURFACE | pygame.DOUBLEBUF
)
clock = pygame.time.Clock()

print(window.get_size())
print(pygame.display.Info())

prev_keys = set()

env.reset()

gamepad = devices.gamepads[0]

STATE = {
    "UP": 0, 
    "DOWN": 0, 
    "LEFT": 0, 
    "RIGHT": 0,
    "A": 0, 
    "B": 0, 
    "X": 0, 
    "START": 0,
    "CAM_X": 0,
    "CAM_Y": 0,
    "LT": 0,
    "RT": 0,
    "L3": 0,
}

lock = threading.Lock()

DEADZONE = 10000
NORM = 32768

def input_loop():
    while True:
        for e in gamepad.read():
            with lock:
                if e.code == "ABS_HAT0X":
                    STATE["LEFT"]  = 1 if e.state == -1 else 0
                    STATE["RIGHT"] = 1 if e.state == 1 else 0

                elif e.code == "ABS_HAT0Y":
                    STATE["UP"]   = 1 if e.state == -1 else 0
                    STATE["DOWN"] = 1 if e.state == 1 else 0

                elif e.code == "ABS_X":
                    if e.state > DEADZONE:
                        STATE["LEFT"] = 0
                        STATE["RIGHT"] = 1
                    elif e.state < -DEADZONE:
                        STATE["LEFT"] = 1
                        STATE["RIGHT"] = 0
                    else:
                        STATE["LEFT"] = 0
                        STATE["RIGHT"] = 0

                elif e.code == "ABS_Y":
                    if e.state > DEADZONE:
                        STATE["UP"] = 1
                        STATE["DOWN"] = 0
                    elif e.state < -DEADZONE:
                        STATE["UP"] = 0
                        STATE["DOWN"] = 1
                    else:
                        STATE["UP"] = 0
                        STATE["DOWN"] = 0

                # RIGHT STICK X
                elif e.code == "ABS_RX":
                    if abs(e.state) > DEADZONE:
                        STATE["CAM_X"] = e.state / NORM
                    else:
                        STATE["CAM_X"] = 0

                # RIGHT STICK Y
                elif e.code == "ABS_RY":
                    if abs(e.state) > DEADZONE:
                        STATE["CAM_Y"] = e.state / NORM
                    else:
                        STATE["CAM_Y"] = 0

                elif e.code == "BTN_SOUTH":
                    STATE["A"] = e.state

                elif e.code == "BTN_EAST":
                    STATE["B"] = e.state

                elif e.code == "BTN_WEST":
                    STATE["X"] = e.state

                elif e.code == "BTN_START":
                    STATE["START"] = e.state

                elif e.code == "BTN_THUMBL":
                    STATE["L3"] = e.state

                elif e.code == "ABS_Z":
                    value = e.state / 255.0
                    if value < 0.5:
                        value = 0
                    STATE["LT"] = value
                
                elif e.code == "ABS_RZ":
                    value = e.state / 255.0
                    if value < 0.5:
                        value = 0
                    STATE["RT"] = value

DEAD_NORM = DEADZONE / NORM

def discretize(value):

    # neutral
    if abs(value) <= DEAD_NORM:
        return 0

    # positive
    if value > 0:
        if value < DEAD_NORM + 0.5:
            return 1
        else:
            return 2

    # negative
    else:
        if value > -(DEAD_NORM + 0.5):
            return 3
        else:
            return 4

t = threading.Thread(target=input_loop, daemon=True)
t.start()

is_running = False
is_recording = False

color=(0, 0, 255)

count_record = 0

trajectories = []
# Data to store
recorded_obs = []
recorded_actions = []


NUMBER_OF_ACTIONS=18

# TODO: FRAME SKIP
COUNT_FRAME = 0

while True:
    pygame.event.pump()
    
    action = [np.zeros(NUMBER_OF_ACTIONS, dtype=np.float32)]

    if keyboard.is_pressed('k'):
        is_running = not is_running
        
        time.sleep(0.5)

        print("k was pressed ", is_running)

    if is_running:
        color=(0, 255, 0)
    else:
        color=(0, 0, 255)

    with lock:
        received_action = [
            STATE["UP"], # 0 
            STATE["DOWN"], 
            STATE["LEFT"], 
            STATE["RIGHT"],
            STATE["A"], 
            STATE["B"], 
            STATE["X"],
            STATE["LT"], # 7
            STATE["RT"],
            STATE["L3"],
            STATE["CAM_X"], # 10
            STATE["CAM_Y"], # 11
        ]

        if keyboard.is_pressed('up') or received_action[0]: action[0][0] = 1
        if keyboard.is_pressed('down') or received_action[1]: action[0][1] = 1
        if keyboard.is_pressed('left') or received_action[2]: action[0][2] = 1
        if keyboard.is_pressed('right') or received_action[3]: action[0][3] = 1
        if keyboard.is_pressed('i') or received_action[4]: action[0][4] = 1
        if keyboard.is_pressed('o') or received_action[5]: action[0][5] = 1
        if keyboard.is_pressed('p') or received_action[6]: action[0][6] = 1
        
        if received_action[7]: action[0][7] = 1
        if received_action[8]: action[0][8] = 1
        if received_action[9]: action[0][9] = 1

        # right analog
        # convert to discrete action
        # 10 11 12 13 <---- right x axis, neutral is [0, 0, 0, 0]
        # 14 15 16 17 <---- right y axis, neutral is [0, 0, 0, 0]
        if received_action[10]:  # X axis
            disc_x = discretize(received_action[10])
            if disc_x == 1:
                action[0][10] = 1
            elif disc_x == 2:
                action[0][11] = 1
            elif disc_x == 3:
                action[0][12] = 1
            elif disc_x == 4:
                action[0][13] = 1
        
        if received_action[11]:  # Y axis
            disc_y = discretize(received_action[11])
            if disc_y == 1:
                action[0][14] = 1
            elif disc_y == 2:
                action[0][15] = 1
            elif disc_y == 3:
                action[0][16] = 1
            elif disc_y == 4:
                action[0][17] = 1

    if is_running != is_recording:
        if is_running:
            print("first obs")
            recorded_obs.append(obs)
            COUNT_FRAME = 0

    # obs, _, _, _, _ = env.step(action)
    obs, _, _, _ = env.step(action)

    img = myEnv.render()

    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

        # img = cv2.resize(img, (128, 128))
        img = cv2.resize(img, (SCREEN_WIDTH, SCREEN_HEIGHT))
        
        cv2.circle(img, center=(100, 100), radius=50, color=color, thickness=2)
        
        text_to_display = "Count: " + str(count_record)
        position = (50, 250)
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 1.5
        thickness = 2
        line_type = cv2.LINE_AA


        cv2.putText(img, text_to_display, position, font, font_scale, color, thickness, line_type)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        surface = pygame.image.frombuffer(
            img,
            (img.shape[1], img.shape[0]),
            "RGB"
        )

        window.blit(surface, (0, 0))
        
    pygame.display.flip()

    received_action = np.zeros(NUMBER_OF_ACTIONS, dtype=np.float32)

    if is_running != is_recording:
        if not is_running:
            print("end recording, increase count")
            count_record = count_record + 1

            # print(recorded_obs[0])

            obs_uint8 = np.stack(
                [obs.astype(np.uint8) for obs in recorded_obs],
                axis=0
            )
            print(obs_uint8.shape)

            trajectories.append(
                Trajectory(
                    obs=obs_uint8,
                    acts=np.array(recorded_actions), 
                    infos = None, 
                    terminal = False
                )
            )

            recorded_obs = []
            recorded_actions = []

    is_recording = is_running

    if is_running:
        # frame skip
        if COUNT_FRAME % 4 == 0 and COUNT_FRAME > 0:
            recorded_obs.append(obs)
            recorded_actions.append(action)
        
        COUNT_FRAME = COUNT_FRAME + 1

    if count_record > MAX_TRAJ - 1:
        print("end recording")
        break

print(len(trajectories))

th.save(trajectories, demo_path + f"demos{last_index + 1}.pt")

pygame.quit()

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


last_index: 16
132968
Window founded HWND: 132968.
(1920, 1440)
<VideoInfo(hw = 0, wm = 1,video_mem = 0
         blit_hw = 0, blit_hw_CC = 0, blit_hw_A = 0,
         blit_sw = 0, blit_sw_CC = 0, blit_sw_A = 0,
         bitsize  = 32, bytesize = 4,
         masks =  (0x00ff0000, 0x0000ff00, 0x000000ff, 0x00000000),
         shifts = (16, 8, 0, 0),
         losses =  (0, 0, 0, 8),
         current_w = 1920, current_h = 1440
         pixel_format = SDL_PIXELFORMAT_RGB888)
>
k was pressed  True
first obs
k was pressed  False
end recording, increase count
(1298, 1, 128, 128, 4)
k was pressed  True
first obs
k was pressed  False
end recording, increase count
(975, 1, 128, 128, 4)
k was pressed  True
first obs
k was pressed  False
end recording, increase count
(1283, 1, 128, 128, 4)
k was pressed  True
first obs
k was pressed  False
end recording, increase count
(1272, 1, 128, 128, 4)
k was pressed  True
first obs
k was pressed  False
end recording, increase count
(1235, 1, 128, 128, 4)
k was

Exception in thread Thread-6 (input_loop):
Traceback (most recent call last):
  File "D:\Python311\Lib\threading.py", line 1038, in _bootstrap_inner
    self.run()
  File "D:\Python311\Lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "D:\Python311\Lib\threading.py", line 975, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\paulo\AppData\Local\Temp\ipykernel_17596\2535252777.py", line 102, in input_loop
  File "D:\Python311\Lib\site-packages\inputs.py", line 2517, in read
    return next(iter(self))
           ^^^^^^^^^^^^^^^^
  File "D:\Python311\Lib\site-packages\inputs.py", line 2686, in __iter__
    self.__check_state()
  File "D:\Python311\Lib\site-packages\inputs.py", line 2695, in __check_state
    raise UnpluggedError(
inputs.UnpluggedError: Gamepad 0 is not connected
